In [2]:
import mlflow
import pandas as pd
import numpy as np
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import r2_score, accuracy_score, confusion_matrix, mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error, precision_score, recall_score, roc_auc_score, f1_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
import imblearn
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

In [2]:
df = pd.read_csv("cleaned_emi_prediction_dataset.csv")

In [3]:
# feature engineering- add some calculated featires
df['total_expense'] = (df['monthly_rent'] + 
                       df['college_fees'] + 
                       df['school_fees'] +
                       df['travel_expenses'] +
                       df['groceries_utilities'] + 
                       df['other_monthly_expenses'])
df['expense_ratio'] = df['total_expense'] / df['monthly_salary']
df['emi_burden'] = df['current_emi_amount'] / df['monthly_salary']

In [4]:
X = df.drop(columns=['emi_eligibility', 'max_monthly_emi'])
y_reg = df['max_monthly_emi']
y_class = df['emi_eligibility']

In [6]:
df.shape

(391003, 30)

In [7]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi,total_expense,expense_ratio,emi_burden
0,38,F,Married,Professional,82600,Private,0.9,Mid-size,Rented,20000.0,...,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0,59900.0,0.725182,0.286925
1,38,F,Married,Graduate,21500,Private,7.0,MNC,Family,0.0,...,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0,15400.0,0.716279,0.190698
2,38,F,Married,Professional,86100,Private,5.8,Startup,Own,0.0,...,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0,35600.0,0.413473,0.000000
3,58,F,Married,High School,66800,Private,2.2,Mid-size,Own,0.0,...,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0,37400.0,0.559880,0.000000
4,48,F,Married,Professional,57300,Private,3.4,Mid-size,Family,0.0,...,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0,58600.0,1.022688,0.000000


In [16]:
# test-train split
X_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=11)

In [17]:
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_reg_final = encoder(df=X_train_reg)
x_test_reg_final = encoder(df=x_test_reg)

# Regression Models

In [10]:
lr = LinearRegression()
lr.fit(X_train_reg_final, y_train_reg)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [11]:
lr.coef_

array([ 2.46227304e+00, -2.26705426e+01, -7.38755328e+00,  4.19927140e+02,
        2.16020762e-02,  6.78125630e+02,  3.68546511e+01,  8.84124676e+00,
        5.08879936e+02, -2.59207260e-01, -2.22428693e+01, -2.22428693e+01,
       -2.81499455e-01, -2.30595064e-01,  3.49624909e-01,  3.58488199e-01,
        4.31259256e-02, -1.89675410e+03, -4.00239697e-01,  7.81988907e+00,
        1.05096094e-02,  4.18665674e-04,  1.82504888e+00, -3.81445875e-05,
        7.19497635e-01, -2.00627380e-02, -3.58935008e+02,  3.34182111e+03])

In [12]:
pred = lr.predict(x_test_reg_final)

In [13]:
r2_score(y_test_reg, pred)

0.7151153105612955

In [14]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    n_jobs=-1
)

In [15]:
rf.fit(X_train_reg_final, y_train_reg)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [16]:
pred = rf.predict(x_test_reg_final)

In [17]:
r2_score(y_test_reg, pred)

0.9842771958874394

In [18]:
root_mean_squared_error(y_test_reg, pred)

978.7123154898212

In [28]:
xgb = XGBRegressor(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    tree_method='hist'
)
xgb.fit(X_train_reg_final, y_train_reg)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_met

In [29]:
pred = xgb.predict(x_test_reg_final)

In [30]:
r2_score(y_test_reg, pred)

0.9914935533586049

In [31]:
mean_squared_error(y_test_reg, pred)

518236.8429001995

In [32]:
root_mean_squared_error(y_test_reg, pred)

719.8866875420044

In [33]:
mean_absolute_percentage_error(y_test_reg, pred)

0.07725990751256916

In [34]:
score = cross_val_score(xgb, X_train_reg_final, y_train_reg, cv=5, scoring='r2')
print(f"CV R2 score: {score.mean()}")

CV R2 score: 0.9903686326358401


# Classification Models

In [5]:
X_train_class, x_test_class,  y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=11)

In [6]:
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_class_encoded = encoder(df=X_train_class)
x_test_class_encoded = encoder(df=x_test_class)

In [53]:
X_train_class_encoded.shape

(312802, 28)

In [41]:
log = LogisticRegression()
log.fit(X_train_class_encoded, y_train_class)

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [52]:
log.coef_

array([[-2.22625218e-07, -1.19842257e-11, -2.07328136e-09,
         5.93298131e-09,  2.66864663e-06, -6.73077923e-10,
         1.61121188e-08, -1.59358906e-08,  5.69040998e-09,
        -4.59675091e-05, -2.01399322e-08, -1.33164155e-08,
        -5.52057545e-05, -5.14067926e-05,  4.69647205e-05,
         7.58448446e-05,  2.45887716e-05, -1.42349607e-08,
        -8.47890883e-05, -2.46677405e-06,  1.68848333e-06,
        -1.29592353e-07, -4.61430321e-09, -5.16085826e-06,
         1.01728565e-06, -5.18171946e-06, -1.09148837e-08,
        -2.46553252e-09],
       [-5.67751724e-07, -6.29555851e-11, -4.76131462e-09,
        -9.31697159e-09,  3.47889881e-07, -1.57066565e-08,
        -6.86938997e-08, -3.64447210e-08, -1.45146868e-08,
        -1.61103185e-06, -4.15594214e-08, -2.65562252e-08,
        -2.91632035e-05, -1.55207013e-05,  4.29659068e-06,
         1.20549182e-05,  5.69353078e-06, -8.37106993e-09,
        -2.99683525e-05, -9.91757794e-06, -2.09370273e-07,
         1.28971694e-06, -3.08

In [42]:
pred = log.predict(x_test_class)

In [43]:
accuracy_score(y_test_class, pred)

0.8583266198641961

In [46]:
precision_score(y_test_class, pred, average='weighted')

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.8195622960253914

In [48]:
f1_score(y_test_class, pred, average='weighted')

0.8383998482204758

In [51]:
recall_score(y_test_class, pred, average='weighted')

0.8583266198641961

In [54]:
rf_class = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
)
rf_class.fit(X_train_class_encoded, y_train_class)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [55]:
pred = rf_class.predict(x_test_class_encoded)

In [56]:
accuracy_score(y_test_class, pred)

0.9074692139486707

In [60]:
precision_score(y_test_class, pred, average='weighted', zero_division=0)

0.8671826030542599

In [58]:
f1_score(y_test_class, pred, average='weighted')

0.8859042033066385

In [59]:
recall_score(y_test_class, pred, average='weighted')

0.9074692139486707

In [7]:
lc = LabelEncoder()
y_train_class_encoded = lc.fit_transform(y_train_class)
y_test_class_encoded = lc.transform(y_test_class)

In [93]:
xgb_class = XGBClassifier(
    n_estimators=50,
    max_depth=4,
    n_jobs=-1
)

In [113]:
xgb_class.fit(X_train_class_encoded, y_train_class_encoded)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [114]:
pred = xgb_class.predict(x_test_class_encoded)

In [123]:
pred_prob = xgb_class.predict_proba(x_test_class_encoded)

In [115]:
pred

array([0, 2, 2, ..., 2, 0, 0], shape=(78201,))

In [127]:
roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')

np.float64(0.9776399024160394)

In [116]:
accuracy_score(y_test_class_encoded, pred)

0.9463689722637818

In [117]:
f1_score(y_test_class_encoded, pred, average='weighted')

0.9256552467603655

In [118]:
recall_score(y_test_class_encoded, pred, average='weighted')

0.9463689722637818

In [119]:
precision_score(y_test_class_encoded, pred, average='weighted')

0.9088698539983783

In [104]:
models_reg = [
    (
        "Linear Regression Model",
        {},
        LinearRegression(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),

    (
        "Random Forest Regressor",
        {"n_estimators":50, "max_depth":15, "n_jobs":-1},
        RandomForestRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),

    (
        "XGBoost Regressor",
        {"n_estimators":50, "max_depth":15, "n_jobs":-1, "tree_method":'hist'},
        XGBRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),
]

models_class = [
    (
        "Logistic Regression",
        {},
        LogisticRegression(),
        (X_train_class_encoded, y_train_class),
        (x_test_class_encoded,y_test_class)
    ),

    (
        "Random Forest Classifier",
        {"n_estimators":50, "max_depth":10, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class),
        (x_test_class_encoded,y_test_class)
    ),

    (
        "XGBoost Classifier",
        {"n_estimators":50, "max_depth":10, "n_jobs":-1},
        XGBClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded,y_test_class_encoded)
    )
]

In [133]:
report_reg = []
for model_name, params, model, train_set, test_set in models_reg:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)

    # calculate metrics
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)

    # store the metrics
    report_reg.append((model_name, rmse, mae, r2, mape))


In [138]:
report_class = []
for model_name, params, model, train_set, test_set in models_class:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    report_class.append((model_name, accuracy, precision, recall, f1, roc_auc))

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [135]:
report_reg

[('Linear Regression Model',
  4166.051230203868,
  2982.77293687861,
  0.7151153105612955,
  1.9078515904236613),
 ('Random Forest Regressor',
  986.7867624263866,
  420.56152853498315,
  0.9840166972077898,
  0.07634307164123287),
 ('XGBoost Regressor',
  840.0317646161602,
  311.7954499368795,
  0.9884172598229413,
  0.06430872045930132)]

In [139]:
report_class

[('Logistic Regression',
  0.8583266198641961,
  0.8195622960253914,
  0.8583266198641961,
  0.8383998482204758,
  0.8391620342046711),
 ('Random Forest Classifier',
  0.9093234101865705,
  0.8690920483259407,
  0.9093234101865705,
  0.8877770670003182,
  0.9385955363754126),
 ('XGBoost Classifier',
  0.9615861689748213,
  0.9559533526666045,
  0.9615861689748213,
  0.9567410556734424,
  0.9883853147328946)]

In [111]:
mlflow.set_experiment("First Regression Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(models_reg):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = report_reg[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"RMSE": report[1],
                            "MAE": report[2],
                            "R2": report[3],
                            "MAPE": report[4]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/03/29 10:39:14 INFO mlflow.tracking.fluent: Experiment with name 'First Regression Experiment' does not exist. Creating a new experiment.
2026/03/29 10:39:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 10:39:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/29 10:39:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 10:39:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code dur

🏃 View run Linear Regression Model at: http://127.0.0.1:5000/#/experiments/1/runs/bf496f62fe71488faa650da1ca290291
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/29 10:39:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/1/runs/0cf7d169d23f46fbb7ab73b972b75e43
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/1/runs/bd62c0f49937425cb356414854ba3311
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [140]:
mlflow.set_experiment("First Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(models_class):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = report_class[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"Accuracy": report[1],
                            "Precision": report[2],
                            "Recall": report[3],
                            "F1": report[4],
                            "ROC_AUC": report[5]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/03/29 11:19:41 INFO mlflow.tracking.fluent: Experiment with name 'First Classification Experiment' does not exist. Creating a new experiment.
2026/03/29 11:19:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 11:19:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/29 11:19:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 11:19:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code

🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/35d4401217594a02b26ba9dae433b694
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/03/29 11:19:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/2/runs/2bb79b823ce44847b3d2683c7cc66973
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/2/runs/20c30b3eaf03446999e85c03e866fc3c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Our first experiments had unscaled data for linear regression and logistic regression, which is not the correct way to train these models.
### And for logistic regression, we've to handle imbalance in data
### Also, we have not tuned hyper-parameters.
### Let's do what we have not done.

## Re-read the data for updated models

In [ ]:
# Data Loading and Required Preprocessing and Feature Engineering
df = pd.read_csv("cleaned_emi_prediction_dataset.csv")

# feature engineering- add some calculated featires
df['total_expense'] = (df['monthly_rent'] + 
                       df['college_fees'] + 
                       df['school_fees'] +
                       df['travel_expenses'] +
                       df['groceries_utilities'] + 
                       df['other_monthly_expenses'])
df['expense_ratio'] = df['total_expense'] / df['monthly_salary']
df['emi_burden'] = df['current_emi_amount'] / df['monthly_salary']

# setting feature and target
X = df.drop(columns=['emi_eligibility', 'max_monthly_emi'])
y_reg = df['max_monthly_emi']
y_class = df['emi_eligibility']

# Regression Split
X_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=11)
# Encoding
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_reg_final = encoder(df=X_train_reg)
x_test_reg_final = encoder(df=x_test_reg)

# Scaling for Linear Regression model
scaler = StandardScaler()
X_train_reg_final_scaled = scaler.fit_transform(X_train_reg_final)
x_test_reg_final_scaled = scaler.transform(x_test_reg_final)

In [ ]:
# Classification Split
X_train_class, x_test_class,  y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=11)

# Encoding
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_class_encoded = encoder(df=X_train_class)
x_test_class_encoded = encoder(df=x_test_class)

# Target Encoding for calssification Model
lc = LabelEncoder()
y_train_class_encoded = lc.fit_transform(y_train_class)
y_test_class_encoded = lc.transform(y_test_class)

In [ ]:
# Imabalance handling using SMOTE for classification models
smote = SMOTE(random_state=11)
X_train_class_encoded, y_train_class_encoded = smote.fit_resample(X_train_class_encoded, y_train_class_encoded)

In [ ]:
# scaling data for Logistic regression

scaler = StandardScaler()
X_train_class_encoded_scaled = scaler.fit_transform(X_train_class_encoded)
x_test_class_encoded_scaled = scaler.transform(x_test_class_encoded)

## Updated Regression Models

In [18]:
# Scale data for Linear Regression

scaler = StandardScaler()
X_train_reg_final_scaled = scaler.fit_transform(X_train_reg_final)
x_test_reg_final_scaled = scaler.transform(x_test_reg_final)


In [146]:
lr_scaled = LinearRegression()
lr_scaled.fit(X_train_reg_final_scaled, y_train_reg)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [147]:
pred = lr_scaled.predict(x_test_reg_final_scaled)

In [151]:
r2_score(y_test_reg, pred)

0.7151153105612967

In [ ]:
param_grid_rf_reg = {
    "n_estimators" : [50,60,70],
    "max_depth" : [10,15,20],
    "n_jobs" : [-1]
}

grid_search = GridSearchCV(RandomForestRegressor(), param_grid_rf_reg, cv=5, scoring='r2')
grid_search.fit(X_train_reg_final, y_train_reg)

print(grid_search.best_score_)
grid_search.best_params_

0.9848897578438948


{'max_depth': 20, 'n_estimators': 60, 'n_jobs': -1}

In [ ]:
param_grid_xgb_reg = {
    "n_estimators" : [50,70,100],
    "max_depth" : [10,15],
    "tree_method" : ['hist'],
    "n_jobs" : [-1]
}

grid_search = GridSearchCV(XGBRegressor(), param_grid_xgb_reg, cv=5, scoring='r2')
grid_search.fit(X_train_reg_final, y_train_reg)

print(grid_search.best_score_)
grid_search.best_params_

0.9906260365773021


{'max_depth': 10, 'n_estimators': 100, 'n_jobs': -1, 'tree_method': 'hist'}

## Updated Classification Models

In [8]:
# since we have imbalane present in our data, we need to handle it for classification models

smote = SMOTE(random_state=11)
X_train_class_encoded, y_train_class_encoded = smote.fit_resample(X_train_class_encoded, y_train_class_encoded)

In [9]:
X_train_class_encoded.shape

(725130, 28)

In [20]:
# scaling data for logistic regression

scaler = StandardScaler()
X_train_class_encoded_scaled = scaler.fit_transform(X_train_class_encoded)
x_test_class_encoded_scaled = scaler.transform(x_test_class_encoded)

In [40]:
log_scaled = LogisticRegression()
log_scaled.fit(X_train_class_encoded_scaled, y_train_class_encoded)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [41]:
p = log_scaled.predict(x_test_class_encoded_scaled)

In [42]:
p

array([0, 2, 2, ..., 2, 0, 0], shape=(78201,))

In [ ]:
accuracy_score(y_test_class_encoded, p)

0.8172401887443894

In [32]:
rf_class = RandomForestClassifier(
    n_estimators=250,
    max_depth=20,
    n_jobs=-1,
)
rf_class.fit(X_train_class_encoded, y_train_class_encoded)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [33]:
p2 = rf_class.predict(x_test_class_encoded)

In [34]:
accuracy_score(y_test_class_encoded, p2)

0.9161008171251007

In [49]:
recall_score(y_test_class_encoded, p2, average='weighted')

0.9324305315788801

In [50]:
f1_score(y_test_class_encoded, p2, average='weighted')

0.9300808324991419

In [51]:
precision_score(y_test_class_encoded, p2, average='weighted')

0.9280562251510299

In [ ]:
# param_grid_rf_class = {
#     "n_estimators" : [50,60],
#     "max_depth" : [16,18]
# }

# grid_search = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid_rf_class, cv=5, scoring='accuracy')
# grid_search.fit(X_train_class_encoded_scaled, y_train_class_encoded)

# print(grid_search.best_score_)
# grid_search.best_params_

# output for this cell when it was run
# 0.9395041940265194
# {'max_depth': 18, 'n_estimators': 60}

0.9395041940265194


{'max_depth': 18, 'n_estimators': 60}

In [ ]:
# Since GridSearchCV was giving some memory allocation error, we did the hyper parameter tunig manually in the above cells.
# Our best marameters are: n_estimators = 250, max_dep = 40, after these params, there was no significant improvement in accuracy

In [14]:
# Same for the XGB Classifier, with manual tuning, we got best estimators:
# n_estimators = 150, max_depth = 15, tree_method = 'hist'

In [11]:
xgb_class = XGBClassifier(
    n_estimators=150,
    max_depth = 15,
    tree_method = 'hist',
    n_jobs=-1
)

xgb_class.fit(X_train_class_encoded, y_train_class_encoded)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [12]:
pred = xgb_class.predict(x_test_class_encoded)

In [13]:
accuracy_score(y_test_class_encoded, pred)

0.9565862329126226

# Implemting MLflow for final models

In [7]:
# Final Models

# In regression models we are now using scaled data and using tuned hyper-parameters
final_models_reg = [
    (
        "Linear Regression Model",
        {},
        LinearRegression(),
        (X_train_reg_final_scaled, y_train_reg),
        (x_test_reg_final_scaled, y_test_reg)
    ),  # In Linear Regression Model, we have updated the model with scaled feature data

    (
        "Random Forest Regressor",
        {"n_estimators":60, "max_depth":20, "n_jobs":-1},
        RandomForestRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),  # In Random Forest Regressor Model, we have tuned for hyper-parameters

    (
        "XGBoost Regressor",
        {"n_estimators":100, "max_depth":10, "n_jobs":-1, "tree_method":'hist'},
        XGBRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),  # In XGBoost Regressor Model, we have tuned for hyper-parameters
]

# In classification models we are using balanced and scaled data and using tuned hyper-parameters 
# (scaled data only for logit, rf and xgb dont need scaled data)
final_models_class = [
    (
        "Logistic Regression",
        {},
        LogisticRegression(),
        (X_train_class_encoded_scaled, y_train_class_encoded),
        (x_test_class_encoded_scaled, y_test_class_encoded)
    ),  # In Logistic Regression Model, we have updated the model with balanced and scaled data

    (
        "Random Forest Classifier",
        {"n_estimators":100, "max_depth":20, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded, y_test_class_encoded)
    ),  # In Random Forest Classifier Model, we have now used balanced data and best hyper-parameters

    (
        "XGBoost Classifier",
        {"n_estimators":150, "max_depth":15, "n_jobs":-1, "tree_method":'hist'},
        XGBClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded,y_test_class_encoded)
    )   # In XGBoost Classifier Model, we have now used balanced data and best hyper-parameters
]

## Training loops and scoring metrics

In [8]:
final_report_reg = []
for model_name, params, model, train_set, test_set in final_models_reg:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)

    # calculate metrics
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)

    # store the metrics
    final_report_reg.append((model_name, rmse, mae, r2, mape))


In [9]:
final_report_reg

[('Linear Regression Model',
  4166.05123020386,
  2982.772936878257,
  0.7151153105612967,
  1.9078515904229363),
 ('Random Forest Regressor',
  929.7760703123067,
  360.24757814083694,
  0.9858101884997351,
  0.06771540082435022),
 ('XGBoost Regressor',
  706.398300691002,
  283.55018688489446,
  0.9918093345228486,
  0.07528680359944108)]

In [12]:
# save regression metrics
columns = ["Model", "RMSE", "MAE", "R2", "MAPE"]
reg_met_df = pd.DataFrame(final_report_reg, columns=columns)
reg_met_df.to_csv("reg_metrics.csv", index=False)

In [10]:
final_report_class = []
for model_name, params, model, train_set, test_set in final_models_class:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    final_report_class.append((model_name, accuracy, precision, recall, f1, roc_auc))

In [14]:
# save classification metrics
columns = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
class_met_df = pd.DataFrame(final_report_class, columns=columns)
class_met_df.to_csv("class_metrics.csv", index=False)

In [11]:
final_report_class

[('Logistic Regression',
  0.8172401887443894,
  0.8889676813505716,
  0.8172401887443894,
  0.8477084132936169,
  0.895298808736541),
 ('Random Forest Classifier',
  0.9141315328448485,
  0.9288521516477589,
  0.9141315328448485,
  0.9207195618114948,
  0.9586858403985773),
 ('XGBoost Classifier',
  0.9565862329126226,
  0.9548879615378755,
  0.9565862329126226,
  0.9556813597766846,
  0.9864819741170519)]

### update to mlflow

In [ ]:
mlflow.set_experiment("Second Regression Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(final_models_reg):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_reg[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"RMSE": report[1],
                            "MAE": report[2],
                            "R2": report[3],
                            "MAPE": report[4]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/03/29 23:53:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 23:53:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/29 23:53:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 23:53:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

🏃 View run Linear Regression Model at: http://127.0.0.1:5000/#/experiments/3/runs/d0f475742da144358f19994da1410551
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/29 23:54:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/3/runs/bc50ce04ac2f4480b4ed41da727ceb33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/3/runs/01ea1cbde8f34dc49ccc386754077b6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [29]:
mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(final_models_class):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_class[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"Accuracy": report[1],
                            "Precision": report[2],
                            "Recall": report[3],
                            "F1": report[4],
                            "ROC_AUC": report[5]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model", pip_requirements=[])
        elif model_name=="Random Forest Classifier":
            mlflow.sklearn.log_model(model, "model", pip_requirements=[])
        else:
            mlflow.sklearn.log_model(model, "model")

2026/03/30 00:19:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 00:19:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/30 00:19:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/4/runs/d81b24dfc9564d6e9e6857f0902ed2d4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/03/30 00:19:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/26f62b52bb894f77bbfefc2c3d4fe7ef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow-artifacts/artifacts/4/models/m-5422df9172a449c0a1c6dfd0c8720ea7/artifacts/model.pkl failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/4/models/m-5422df9172a449c0a1c6dfd0c8720ea7/artifacts/model.pkl (Caused by ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)))

In [ ]:
# Since previously, random forest classification model was not getting logged because of high hyper-parameters. 
# We reduced the hyper-parameters while keeping the accuracy more than 90% or 0.90.
# And then logged RF Classifier and XGB Classifier one by one manually

In [ ]:
# rf model alone
rf_model = [
    (
        "Random Forest Classifier",
        {"n_estimators":100, "max_depth":20, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded, y_test_class_encoded)
    ),  # In Random Forest Classifier Model, we have now used balanced data and best hyper-parameters
]

In [20]:
report_rf = []
for model_name, params, model, train_set, test_set in rf_model:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    report_rf.append((model_name, accuracy, precision, recall, f1, roc_auc))

In [21]:
report_rf

[('Random Forest Classifier',
  0.914182683085894,
  0.9295104717615641,
  0.914182683085894,
  0.9209967590612343,
  0.9595573442831848)]

In [ ]:
# Log RF Classifier

mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(rf_model):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = report_rf[i]
    if model_name=="Random Forest Classifier":
        with mlflow.start_run(run_name=model_name):
            mlflow.log_params(params)
            mlflow.log_metrics({"Accuracy": report[1],
                                "Precision": report[2],
                                "Recall": report[3],
                                "F1": report[4],
                                "ROC_AUC": report[5]})
            
            # if "XGB" in model_name:
                # mlflow.xgboost.log_model(model, "model", pip_requirements=[])
            if model_name=="Random Forest Classifier":
                mlflow.sklearn.log_model(model, "model", pip_requirements=[])
            # else:
                # mlflow.sklearn.log_model(model, "model")

2026/03/30 09:16:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/30 09:16:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/71a3047752d54cceaed761ac7578be0b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [ ]:
# Log XGB Classifier

mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(rf_model):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_class[i]
    if "XGB" in model_name:
        with mlflow.start_run(run_name=model_name):
            mlflow.log_params(params)
            mlflow.log_metrics({"Accuracy": report[1],
                                "Precision": report[2],
                                "Recall": report[3],
                                "F1": report[4],
                                "ROC_AUC": report[5]})
            
            if "XGB" in model_name:
                mlflow.xgboost.log_model(model, "model", pip_requirements=[])
            # if model_name=="Random Forest Classifier":
                # mlflow.sklearn.log_model(model, "model", pip_requirements=[])
            # else:
                # mlflow.sklearn.log_model(model, "model")

2026/03/30 08:27:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/9695d06048224deb81d1f0fa360c2936
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


# Model Registration

In [ ]:
# Linear Regression

model_name = 'Linear Regression Model'
run_id = 'd0f475742da144358f19994da1410551'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'Linear Regression Model' already exists. Creating a new version of this model...
2026/03/30 10:06:45 WARNING mlflow.tracking._model_registry.fluent: Run with id d0f475742da144358f19994da1410551 has no artifacts at artifact path 'model', registering model based on models:/m-ac38a01bc89c46439c2a28edf06f2d17 instead
2026/03/30 10:06:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Linear Regression Model, version 1


🏃 View run Linear Regression Model at: http://127.0.0.1:5000/#/experiments/3/runs/d0f475742da144358f19994da1410551
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '1' of model 'Linear Regression Model'.


In [28]:
# Random Forest Regression

model_name = 'RF Regressor'
run_id = 'bc50ce04ac2f4480b4ed41da727ceb33'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'RF Regressor'.
2026/03/30 10:11:07 WARNING mlflow.tracking._model_registry.fluent: Run with id bc50ce04ac2f4480b4ed41da727ceb33 has no artifacts at artifact path 'model', registering model based on models:/m-4cac947a07694abebc80b98ba9e7c979 instead
2026/03/30 10:11:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF Regressor, version 1


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/3/runs/bc50ce04ac2f4480b4ed41da727ceb33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '1' of model 'RF Regressor'.


In [29]:
# XGBoost Regression

model_name = 'XGBoost Regressor'
run_id = '01ea1cbde8f34dc49ccc386754077b6b'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'XGBoost Regressor'.
2026/03/30 10:12:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 01ea1cbde8f34dc49ccc386754077b6b has no artifacts at artifact path 'model', registering model based on models:/m-7f41ad7c15174813aa0025d1a8575b5a instead
2026/03/30 10:12:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Regressor, version 1
Created version '1' of model 'XGBoost Regressor'.


🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/3/runs/01ea1cbde8f34dc49ccc386754077b6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [ ]:
# Logistic Regression

model_name = 'Logistic Regression'
run_id = '98b1c90307ed487fa9432e6cb3e9bd99'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'Logistic Regression'.
2026/03/30 10:24:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 98b1c90307ed487fa9432e6cb3e9bd99 has no artifacts at artifact path 'model', registering model based on models:/m-88bea93f7eab4e0299894fc5d02b451b instead
2026/03/30 10:24:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regression, version 1


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/4/runs/98b1c90307ed487fa9432e6cb3e9bd99
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


Created version '1' of model 'Logistic Regression'.


In [31]:
# RF Classifier

model_name = 'RF Classifier'
run_id = '71a3047752d54cceaed761ac7578be0b'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'RF Classifier'.
2026/03/30 10:26:45 WARNING mlflow.tracking._model_registry.fluent: Run with id 71a3047752d54cceaed761ac7578be0b has no artifacts at artifact path 'model', registering model based on models:/m-cdbec0c006f24399a098aca8f0df50e9 instead
2026/03/30 10:26:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF Classifier, version 1
Created version '1' of model 'RF Classifier'.


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/71a3047752d54cceaed761ac7578be0b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [32]:
# XGBoost Classifier

model_name = 'XGBoost Classifier'
run_id = '9695d06048224deb81d1f0fa360c2936'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Successfully registered model 'XGBoost Classifier'.
2026/03/30 10:26:52 WARNING mlflow.tracking._model_registry.fluent: Run with id 9695d06048224deb81d1f0fa360c2936 has no artifacts at artifact path 'model', registering model based on models:/m-ccad287acc3f4d52a4607aec3032df94 instead
2026/03/30 10:26:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Classifier, version 1
Created version '1' of model 'XGBoost Classifier'.


🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/9695d06048224deb81d1f0fa360c2936
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
